# Proyecto 5: Procesamiento de Señales y Análisis de Fourier

**Objetivo**: Pipeline de procesamiento de señales: generación, FFT, filtrado, reconstrucción y visualización.

**Features demostradas**:
- `numpy-fast.lp` para generación y operaciones vectorizadas
- FFT con NumPy desde Lisp
- Filtrado pasa-bajos y pasa-altos con scipy
- `defmath` para definir señales compuestas
- Visualización de espectros de frecuencia
- `continuations.lp` para procesamiento streaming con generadores
- `py:->lisp` para traer resultados a Lisp

## 1. Inicialización

In [ ]:
(load "lib/numpy-fast.lp")
(load "lib/continuations.lp")
(py:import "numpy")
(py:import "scipy.signal")
(py:import "matplotlib.pyplot")

## 2. Generación de Señal Compuesta

Creamos una señal con múltiples componentes de frecuencia más ruido.

In [ ]:
(py:run "
import numpy as np

# Parámetros de muestreo
fs = 1000  # Frecuencia de muestreo (Hz)
t = np.linspace(0, 2, 2 * fs, endpoint=False)

# Señal compuesta: 3 sinusoides + ruido
senal = (
    np.sin(2 * np.pi * 5 * t) +            # 5 Hz
    0.5 * np.sin(2 * np.pi * 50 * t) +     # 50 Hz
    0.3 * np.sin(2 * np.pi * 120 * t) +    # 120 Hz
    0.1 * np.random.randn(len(t))           # Ruido
)

print(f'Señal generada: {len(senal)} muestras')
print(f'Frecuencia de muestreo: {fs} Hz')
print(f'Duración: {t[-1]:.2f} s')
print(f'Componentes: 5 Hz, 50 Hz, 120 Hz + ruido')
")

## 3. Análisis de Fourier (FFT)

Calculamos la Transformada Rápida de Fourier para analizar el espectro de frecuencias.

In [ ]:
(py:run "
# FFT
n = len(t)
fft_result = np.fft.fft(senal)
freqs = np.fft.fftfreq(n, 1/fs)

# Magnitud (solo frecuencias positivas)
mask = freqs > 0
magnitudes = np.abs(fft_result[mask]) / n
freqs_pos = freqs[mask]

# Picos principales
from scipy.signal import find_peaks
peaks_idx, _ = find_peaks(magnitudes, height=0.01, distance=5)
print('Frecuencias dominantes detectadas:')
for idx in peaks_idx[:10]:
    print(f'  {freqs_pos[idx]:6.2f} Hz  (magnitud: {magnitudes[idx]:.3f})')
")

## 4. Visualización del Espectro

Graficamos la señal en el dominio del tiempo y la frecuencia.

In [ ]:
(py:run "
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Dominio del tiempo (primeros 500 puntos)
axes[0].plot(t[:500], senal[:500], 'b-', linewidth=0.8)
axes[0].set_xlabel('Tiempo (s)')
axes[0].set_ylabel('Amplitud')
axes[0].set_title('Señal en el Dominio del Tiempo (primeras 500 muestras)')
axes[0].grid(True, alpha=0.3)

# Dominio de la frecuencia
axes[1].plot(freqs_pos[:n//4], magnitudes[:n//4], 'r-', linewidth=0.8)
axes[1].set_xlabel('Frecuencia (Hz)')
axes[1].set_ylabel('Magnitud')
axes[1].set_title('Espectro de Frecuencias (FFT)')
axes[1].grid(True, alpha=0.3)

# Marcar picos
for idx in peaks_idx:
    if freqs_pos[idx] < 200:
        axes[1].annotate(f'{freqs_pos[idx]:.0f} Hz', 
                        (freqs_pos[idx], magnitudes[idx]),
                        textcoords='offset points', xytext=(0,10),
                        ha='center', fontsize=9, color='darkred')

plt.tight_layout()
plt.savefig('analisis-fourier.png', dpi=100)
plt.show()
")

## 5. Diseño de Filtros Digitales

Diseñamos y aplicamos filtros pasa-bajos y pasa-banda con scipy.

In [ ]:
(py:run "
from scipy.signal import butter, filtfilt, freqz

# Filtro Butterworth pasa-bajos (frecuencia de corte: 30 Hz)
def butter_lowpass(cutoff, fs, order=4):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return b, a

def apply_filter(data, cutoff, fs, order=4):
    b, a = butter_lowpass(cutoff, fs, order)
    return filtfilt(b, a, data)

# Aplicar filtros
senal_filtrada_30hz = apply_filter(senal, 30, fs, order=4)
senal_filtrada_80hz = apply_filter(senal, 80, fs, order=4)

print('Filtros aplicados:')
print(f'  Pasa-bajos 30 Hz (elimina 50Hz y 120Hz)')
print(f'  Pasa-bajos 80 Hz (elimina 120Hz, conserva 5Hz y 50Hz)')
")

## 6. Comparación Señal Original vs Filtrada

In [ ]:
(py:run "
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

plot_range = slice(0, 300)  # Primeras 300 muestras para claridad

axes[0].plot(t[plot_range], senal[plot_range], 'b-', linewidth=0.8, alpha=0.7, label='Original')
axes[0].plot(t[plot_range], senal_filtrada_30hz[plot_range], 'r-', linewidth=1.5, label='Filtrada 30 Hz')
axes[0].set_ylabel('Amplitud')
axes[0].set_title('Filtro Pasa-Bajos 30 Hz (conserva solo 5 Hz)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t[plot_range], senal[plot_range], 'b-', linewidth=0.8, alpha=0.7, label='Original')
axes[1].plot(t[plot_range], senal_filtrada_80hz[plot_range], 'g-', linewidth=1.5, label='Filtrada 80 Hz')
axes[1].set_ylabel('Amplitud')
axes[1].set_title('Filtro Pasa-Bajos 80 Hz (conserva 5 Hz y 50 Hz)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Espectros comparados
from scipy.signal import welch
f_orig, pxx_orig = welch(senal, fs, nperseg=256)
f_30, pxx_30 = welch(senal_filtrada_30hz, fs, nperseg=256)
f_80, pxx_80 = welch(senal_filtrada_80hz, fs, nperseg=256)

axes[2].semilogy(f_orig, pxx_orig, 'b-', linewidth=1, alpha=0.7, label='Original')
axes[2].semilogy(f_30, pxx_30, 'r-', linewidth=1.5, label='Filtrada 30 Hz')
axes[2].semilogy(f_80, pxx_80, 'g-', linewidth=1.5, label='Filtrada 80 Hz')
axes[2].set_xlabel('Frecuencia (Hz)')
axes[2].set_ylabel('Densidad Espectral (dB)')
axes[2].set_title('Densidad Espectral de Potencia (Welch)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
axes[2].set_xlim(0, 200)

plt.tight_layout()
plt.savefig('filtros-comparacion.png', dpi=100)
plt.show()
")

## 7. Procesamiento Streaming con Generadores Python

Usamos `continuations.lp` para procesar la señal en streaming.

In [ ]:
;; Crear generador Python que produce muestras en bloques
(py:run "
def streaming_blocks(data, block_size=50):
    '''Generador que emite bloques de la señal.'''
    for i in range(0, len(data), block_size):
        block = data[i:i+block_size]
        energy = np.sum(block ** 2)
        yield {'index': i, 'data': block.tolist(), 'energy': float(energy)}

stream = streaming_blocks(senal, block_size=100)
")

In [ ]:
;; Consumir el generador desde Lisp
(let ((gen @stream)
      (bloques-procesados 0)
      (energia-total 0.0))
  (loop for bloque = (py:gen-next gen)
        while bloque
        do (let* ((idx (py:getitem bloque "index"))
                  (energia (py:getitem bloque "energy")))
             (incf bloques-procesados)
             (incf energia-total energia)
             (when (zerop (mod idx 500))
               (format t "Bloque ~D (muestra ~D): energía = ~,4F~%"
                       bloques-procesados idx energia))))
  (format t "~%Total: ~D bloques procesados~%" bloques-procesados)
  (format t "Energía total de la señal: ~,4F~%" energia-total))

## 8. Análisis Estadístico de la Señal

Extraemos estadísticas a Lisp para análisis.

In [ ]:
;; Calcular estadísticas vía NumPy
(let ((senal-lisp (py:->lisp @senal)))
  (format t "=== Estadísticas de la Señal ===~%")
  (format t "Número de muestras: ~D~%" (length senal-lisp))
  (format t "Mínimo: ~,4F~%" (reduce #'min senal-lisp))
  (format t "Máximo: ~,4F~%" (reduce #'max senal-lisp))
  (let* ((suma (reduce #'+ senal-lisp))
         (n (length senal-lisp))
         (media (/ suma n))
         (varianza (/ (reduce #'+ (mapcar (lambda (x) (expt (- x media) 2)) senal-lisp)) n))
         (std (sqrt varianza)))
    (format t "Media: ~,6F~%" media)
    (format t "Desviación estándar: ~,4F~%" std)
    (format t "RMS: ~,4F~%" (sqrt (/ (reduce #'+ (mapcar (lambda (x) (* x x)) senal-lisp)) n)))))

;; Comparar con NumPy
(py:run "
print(f'\\nVerificación NumPy:')
print(f'  Media: {np.mean(senal):.6f}')
print(f'  Std:   {np.std(senal):.4f}')
print(f'  RMS:   {np.sqrt(np.mean(senal**2)):.4f}')
")

## 9. Detección de Eventos — Cruce por Cero

Algoritmo en Lisp puro para detectar cruces por cero en la señal filtrada.

In [ ]:
;; Detectar cruces por cero
(defun count-zero-crossings (data)
  (let ((count 0))
    (loop for i from 1 below (length data)
          for prev = (nth (- i 1) data)
          for curr = (nth i data)
          when (and (/= prev 0) (/= curr 0)
                    (/= (signum prev) (signum curr)))
          do (incf count))
    count))

(let* ((senal-30hz (py:->lisp @senal_filtrada_30hz))
       (senal-80hz (py:->lisp @senal_filtrada_80hz))
       (original (py:->lisp @senal)))
  (format t "=== Cruces por Cero ===~%")
  (format t "Señal original:             ~D~%" (count-zero-crossings original))
  (format t "Filtrada 30 Hz (solo 5 Hz):  ~D~%" (count-zero-crossings senal-30hz))
  (format t "Filtrada 80 Hz (5+50 Hz):    ~D~%" (count-zero-crossings senal-80hz))
  (format t "~%Esperado 5 Hz: ~D cruces en 2s~%" (* 5 2 2)))

## 10. Ejercicios Propuestos

1. Implementar un filtro notch para eliminar los 50 Hz (frecuencia de red eléctrica)
2. Crear un espectrograma (STFT) para visualizar la evolución temporal de frecuencias
3. Implementar detección de envolvente con la transformada de Hilbert
4. Usar `defmath` para definir una señal FM (frecuencia modulada) y analizar su espectro
5. Implementar un ecualizador paramétrico con sliders interactivos